In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


In [2]:
n = 300
n_categories = 100
customer_id = np.random.randint(0, n_categories, size=n)
size_m2 = np.random.normal(120, 30, n)
noise = np.random.normal(0, 20, n)
 
# price depends ONLY on size_m2 + noise; customer_id has NO real effect
price = 0.8 * size_m2 + noise + 100
df = pd.DataFrame({'customer_id': customer_id, 'size_m2': size_m2, 'price': price})


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
   df[['customer_id', 'size_m2']], df['price'], test_size=0.25, random_state=7 )


In [4]:
full_means = df.groupby('customer_id')['price'].mean()  # peeks at test rows too
X_train_leak = X_train.copy(); X_test_leak = X_test.copy()
X_train_leak['id_encoded'] = X_train_leak['customer_id'].map(full_means)
X_test_leak['id_encoded']  = X_test_leak['customer_id'].map(full_means)
 
model_leak = LinearRegression().fit(X_train_leak[['id_encoded', 'size_m2']], y_train)
pred_leak = model_leak.predict(X_test_leak[['id_encoded', 'size_m2']])
r2_leak = r2_score(y_test, pred_leak)


In [5]:
train_means = X_train.assign(price=y_train).groupby('customer_id')['price'].mean()
X_train_ok = X_train.copy(); X_test_ok = X_test.copy()
X_train_ok['id_encoded'] = X_train_ok['customer_id'].map(train_means)
X_test_ok['id_encoded']  = X_test_ok['customer_id'].map(train_means).fillna(y_train.mean())
 
model_ok = LinearRegression().fit(X_train_ok[['id_encoded', 'size_m2']], y_train)
pred_ok = model_ok.predict(X_test_ok[['id_encoded', 'size_m2']])
r2_ok = r2_score(y_test, pred_ok)


In [6]:
def adjusted_r2(r2, n_obs, n_features):
    return 1 - (1 - r2) * (n_obs - 1) / (n_obs - n_features - 1)
 
n_test = len(y_test); n_features = 2
print(f'Leaky   -> R2: {r2_leak:.4f} | Adj R2: {adjusted_r2(r2_leak, n_test, n_features):.4f}')
print(f'Correct -> R2: {r2_ok:.4f}  | Adj R2: {adjusted_r2(r2_ok, n_test, n_features):.4f}')


Leaky   -> R2: 0.6675 | Adj R2: 0.6583
Correct -> R2: 0.4489  | Adj R2: 0.4336


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
 
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
 
# cross_val_score refits the pipeline INSIDE each fold,
# so scaling/encoding never sees that fold's validation rows.
scores = cross_val_score(pipeline, X_train[['size_m2']], y_train, cv=5, scoring='r2')
print(scores.mean())


0.5334593061546145


In [8]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score


# Generate an imbalanced dataset (95% negative, 5% positive)
X, y = make_classification(
    n_samples=1000, 
    weights=[0.95, 0.05],
    flip_y=0, 
    random_state=42
)


print("Overall class counts:", np.bincount(y))  # Output: [950, 50]


Overall class counts: [950  50]


In [9]:
_, _, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
print(f"Test set positive class ratio (Unstratified): {y_test.mean():.4f}")


Test set positive class ratio (Unstratified): 0.0550


In [10]:
_, _, y_train_s, y_test_s = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
print(f"Test set positive class ratio (Stratified): {y_test_s.mean():.4f}")  # Output: Exactly 0.05


Test set positive class ratio (Stratified): 0.0500


In [11]:
def evaluate_minority_counts(use_stratify: bool) -> np.ndarray:
    minority_counts = []
    for seed in range(1000):
        _, _, _, y_te = train_test_split(
            X, y, test_size=0.2, random_state=seed,
            stratify=y if use_stratify else None
        )
        minority_counts.append(y_te.sum())
    return np.array(minority_counts)


counts_unstratified = evaluate_minority_counts(use_stratify=False)
counts_stratified = evaluate_minority_counts(use_stratify=True)


print("Unstratified Split -> Min:", counts_unstratified.min(), 
      "| Max:", counts_unstratified.max(), 
      "| Std Dev:", counts_unstratified.std())


print("Stratified Split   -> Min:", counts_stratified.min(), 
      "| Max:", counts_stratified.max(), 
      "| Std Dev:", counts_stratified.std())


Unstratified Split -> Min: 2 | Max: 20 | Std Dev: 2.8090409395379057
Stratified Split   -> Min: 10 | Max: 10 | Std Dev: 0.0


In [12]:
def evaluate_model_recall(use_stratify: bool):
    scores = []
    for seed in range(100):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=seed,
            stratify=y if use_stratify else None
        )
        model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
        scores.append(recall_score(y_te, model.predict(X_te)))
    return np.mean(scores), np.std(scores)


mean_unstratified, std_unstratified = evaluate_model_recall(use_stratify=False)
mean_stratified, std_stratified = evaluate_model_recall(use_stratify=True)


print(f"Unstratified Recall -> Mean: {mean_unstratified:.4f}, Std: {std_unstratified:.4f}")
print(f"Stratified Recall   -> Mean: {mean_stratified:.4f}, Std: {std_stratified:.4f}")


Unstratified Recall -> Mean: 0.3032, Std: 0.1366
Stratified Recall   -> Mean: 0.2860, Std: 0.1327
